<a href="https://colab.research.google.com/github/krgyaan/LLM-Finetuning/blob/main/Instruction_finetuning_on_domain_specific_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- Pretrain Model(LLAM)
- Non Instrunction finetuning on plain text
- Instruction finetuning on instrunction data
- then next Preference Alignment



In [ ]:
!pip install -U transformers peft bitsandbytes accelerate

In [ ]:
!pip install -U trl

In [ ]:
!pip install -U PyMuPDF

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, TaskType, get_peft_model
from datasets import load_dataset

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [ ]:
import zipfile
import os

zip_path = "/content/llm-gem-data.zip"

with zipfile .ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall()

In [ ]:
trained_model_path = "/content/checkpoint-4/"

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
nif_trained_model = PeftModel.from_pretrained(base_model, trained_model_path)

In [ ]:
non_instruction_model = AutoModelForCausalLM.from_pretrained(trained_model_path, device_map="auto")

In [ ]:
prompt="Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

inputs = tokenizer(prompt, return_tensors="pt").to(device)
non_instruction_model.to(device)

In [ ]:
outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

print("\nModel Output: \n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Now let's start Instruction Finetuning
- Start with inspecting built in data (Amod/mental_health_counseling_conversations)

In [ ]:
from datasets import Dataset, load_dataset
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")

In [ ]:
dataset

In [ ]:
def format_row(data):
  que = data["Context"]
  ans = data["Response"]
  data["Text"] = f"[INST] {que} [/INST] {ans}"
  return data

In [ ]:
formatted_data = dataset.map(format_row)

In [ ]:
formatted_data[0]["Text"]

In [ ]:
import pandas as pd

# Convert dataset to dataframe
df = pd.DataFrame(dataset)

In [ ]:
# You can convert it into any format now:
# df.to_csv("mental_health_counseling_conversations.csv", index=False)
df.to_json("mental_health_counseling_conversations.json", orient="records", lines=True)

In [ ]:
# And after conversion you can re-load the data like:
from datasets import load_dataset

# dataset_from_csv = load_dataset("csv", data_files="/content/mental_health_counseling_conversations.csv", split="train")
dataset_from_json = load_dataset("json", data_files="/content/mental_health_counseling_conversations.json", split="train")

#### Now let's load our dataset

In [ ]:
from datasets import load_dataset

own_dataset = load_dataset("csv", data_files="/content/pharma_instruction_data.csv", split="train")

In [ ]:
def format_own_data(data):
  ins = data["instruction"]
  inp = data["input"]
  oup = data["output"]
  prompt = f"###Instruction:\n{ins}\n###Input:\n{inp}\n###Output:\n{oup}"
  return {"text": prompt}

In [ ]:
formatted_own_data = own_dataset.map(format_own_data)

In [ ]:
formatted_own_data['text'][0]

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_fn(example):
    tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
tokenized = formatted_own_data.map(tokenize_fn, batched=True)

In [ ]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

non_instruction_model.config.use_cache = False          # Required for gradient checkpointing
non_instruction_model.enable_input_require_grads()      # Required for LoRA + quantized models

In [ ]:
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig, TaskType

model = prepare_model_for_kbit_training(non_instruction_model)   # Must be called before get_peft_model

In [ ]:
# Loaded Quantized Model

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0}        # Force everything onto GPU 0, not "auto"
)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [ ]:
qlora_model = get_peft_model(non_instruction_model, lora_config)
qlora_model.print_trainable_parameters()   # sanity check

In [ ]:
training_args = TrainingArguments(
    output_dir = "./llm-gem-data",    # Save model/checkpoints -> folder path -> where your model ends up
    num_train_epochs = 2,             # Training cycle -> 1-5 -> More = longer training
    per_device_train_batch_size = 2,  # Batch per GPU -> 1-8(Colab) -> Affects speed & memore
    save_steps = 500,                 # Save frequency -> 100-1000 -> More = frequent checkpoints
    save_total_limit = 2,             # Keep last N ->1-3 -> Saves disks
    logging_steps = 50,               # Log every X step -. 20-100  -> for progress monitoring
    learning_rate = 2e-5,             # Weight updates rate -> 1e-5 - 5e-5 ->Controls stability
    fp16 = True,                      # Mixed precision -> True -> Faster + less memory
    report_to = "none",               # Logging destination -> "none" -> Keeps it simple
    optim="paged_adamw_8bit",         # Memory-efficient optimizer for QLoRA
    gradient_checkpointing=True,
)

In [ ]:
trainer = Trainer(
    model=qlora_model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator
)

In [ ]:
trainer.train()

In [ ]:
model_path = "/content/llm-gem-data/checkpoint-6/"

In [ ]:
# Clear the cache and Garbage Collection

import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from peft import PeftModel

instruction_model = PeftModel.from_pretrained(non_instruction_model, model_path)

# model = PeftModel.from_pretrained(
#     base_model,
#     "/content/llm-gem-data/checkpoint-6/"
# )

In [ ]:
instruction_model

In [ ]:
prompt = "List two advantages of combining Atorvastatin with Ezetimibe."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))